# Activity 8 — Apply the trained model to the whole image

**Learning objective:** Use the learned relationship between class labels and satellite features to classify every valid pixel.

This is the step that generates an `Efate_Cleanv4.tif`-type output.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

import odc.geo.xr

## 1. Make sure predictor order matches training order

This is critical. The model must receive raster bands in exactly the same order used during training.

In [ ]:
print("Training feature order:")
print(feature_columns)

missing = [f for f in feature_columns if f not in median.data_vars]

if missing:
    raise ValueError(f"These model features are missing from the raster: {missing}")

predictor_ds = median[feature_columns]

## 2. Stack the image pixels into rows

In [ ]:
stacked = (
    predictor_ds
    .to_array("feature")
    .stack(pixel=["y", "x"])
    .transpose("pixel", "feature")
)

print("Stacked image shape:", stacked.shape)

## 3. Predict only valid pixels

Cloud masking can leave `NaN` values.  
We avoid sending those NoData pixels to `RandomForestClassifier.predict()`.

In [ ]:
stacked_values = stacked.values

valid_pixels = np.all(np.isfinite(stacked_values), axis=1)

print("Total pixels:", len(valid_pixels))
print("Valid pixels:", valid_pixels.sum())
print("NoData pixels:", (~valid_pixels).sum())

In [ ]:
# Keep NaN outside the valid mapped area.
predicted_flat = np.full(
    stacked_values.shape[0],
    np.nan,
    dtype="float32"
)

predicted_flat[valid_pixels] = model.predict(
    stacked_values[valid_pixels]
).astype("float32")

## 4. Reshape predictions back to a map

In [ ]:
predicted_array = predicted_flat.reshape(
    len(predictor_ds.y),
    len(predictor_ds.x)
)

predicted_da = xr.DataArray(
    predicted_array,
    coords={
        "y": predictor_ds.y,
        "x": predictor_ds.x
    },
    dims=["y", "x"],
    name="prediction"
)

# Copy spatial reference information.
predicted_da = predicted_da.odc.assign_crs(
    predictor_ds.odc.geobox.crs
)

print(predicted_da)

In [ ]:
predicted_da.plot.imshow(figsize=(10, 8))
plt.title("Random Forest Predicted Classes")
plt.show()

## 5. Export the prediction raster

In [ ]:
Path("Results").mkdir(exist_ok=True)

OUTPUT_RASTER = "Results/Efate_Invasive_Prediction.tif"

predicted_da.odc.write_cog(
    OUTPUT_RASTER,
    overwrite=True
)

print("Saved:", OUTPUT_RASTER)